In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

prices = pd.read_csv("../data/processed/prices.csv", index_col="date", parse_dates=True)
log_prices = np.log(prices)
log_prices.shape

(731, 134)

In [2]:
# broader "pegged asset" filter: low volatility relative to price level, not just "near $1"
volatility = prices.pct_change(fill_method=None).std()
likely_pegged = volatility[volatility < 0.01].index
KNOWN_PEGGED = list(likely_pegged) + ["euro-coin", "societe-generale-forge-eurcv", "pax-gold", "tether-gold"]
print(f"Excluding {len(KNOWN_PEGGED)} pegged/tracked assets: {KNOWN_PEGGED}")

log_prices_clean = log_prices.drop(columns=[c for c in KNOWN_PEGGED if c in log_prices.columns])

Excluding 14 pegged/tracked assets: ['crvusd', 'dai', 'eurite', 'euro-coin', 'first-digital-usd', 'paypal-usd', 'societe-generale-forge-eurcv', 'tether', 'true-usd', 'usd-coin', 'euro-coin', 'societe-generale-forge-eurcv', 'pax-gold', 'tether-gold']


In [3]:
WINDOW = 180
recent_window = log_prices_clean.iloc[-WINDOW:]

corr_matrix = recent_window.corr()
corr_matrix.shape

(122, 122)

In [4]:
THRESHOLD = 0.9
TOP_N_PER_COIN = 3

pairs = []
for coin in corr_matrix.columns:
    top_matches = corr_matrix[coin].drop(coin).sort_values(ascending=False).head(TOP_N_PER_COIN)
    for match, corr_value in top_matches.items():
        if corr_value >= THRESHOLD:
            pair = tuple(sorted([coin, match]))
            pairs.append((pair[0], pair[1], corr_value))

pairs_df = pd.DataFrame(pairs, columns=["coin_a", "coin_b", "correlation"]).drop_duplicates()
pairs_df = pairs_df.sort_values("correlation", ascending=False)
print(f"{len(pairs_df)} candidate pairs found")
pairs_df.head(15)

139 candidate pairs found


,coin_a,coin_b,correlation
85,hedera-hashgraph,iota,0.981705
118,milk-alliance,the-sandbox,0.981107
79,gas,vechain,0.974084
133,polkadot,the-sandbox,0.973826
94,iota,polkadot,0.972849
82,havven,kusama,0.972742
119,milk-alliance,zetachain,0.971474
80,gas,milk-alliance,0.970599
112,livepeer,the-sandbox,0.969434
146,milk-alliance,skale,0.967954


In [7]:
from statsmodels.tsa.stattools import adfuller
import statsmodels.api as sm

def test_cointegration(price_a, price_b):
    """Returns the ADF p-value on the OLS residual spread, or None if insufficient clean data."""
    combined = pd.concat([price_a, price_b], axis=1).dropna()
    combined = combined[np.isfinite(combined).all(axis=1)]
    if len(combined) < 100:  # need enough points for a meaningful test
        return None
    x = sm.add_constant(combined.iloc[:, 1])
    model = sm.OLS(combined.iloc[:, 0], x).fit()
    spread = model.resid
    return adfuller(spread)[1]

COINT_PVALUE_THRESHOLD = 0.05
selected_pairs = []
skipped_count = 0

for _, row in pairs_df.iterrows():
    a, b = row["coin_a"], row["coin_b"]
    p_value = test_cointegration(recent_window[a], recent_window[b])
    if p_value is None:
        skipped_count += 1
        continue
    if p_value < COINT_PVALUE_THRESHOLD:
        selected_pairs.append({"coin_a": a, "coin_b": b, "correlation": row["correlation"], "p_value": p_value})

selected_pairs_df = pd.DataFrame(selected_pairs).sort_values("p_value")
print(f"{len(selected_pairs_df)} pairs pass both screens ({skipped_count} skipped for insufficient clean data)")
selected_pairs_df

87 pairs pass both screens (0 skipped for insufficient clean data)


,coin_a,coin_b,correlation,p_value
30,havven,jasmycoin,0.956537,7.554043e-07
47,filecoin,havven,0.946300,3.632968e-06
52,ripple,vechain,0.943062,3.760719e-06
48,reserve-rights-token,sushi,0.946061,1.053641e-05
3,milk-alliance,skale,0.967954,1.780441e-05
...,...,...,...,...
80,ethereum-name-service,virtual-protocol,0.913682,3.583746e-02
57,floki,illuvium,0.940793,3.839181e-02
67,livepeer,sei-network,0.931699,3.890398e-02
68,ankr,vechain,0.931639,4.847001e-02


In [9]:
selected_pairs_df.to_csv("../data/processed/selected_pairs.csv", index=False)